# Homework 7: Mạng LSTM (Long Short-Term Memory) để Dự đoán Tên
## Môn học: Trí Tuệ Nhân Tạo - EE3063

Nhóm Thực Hiện: [Điền tên thành viên nhóm]
MSSV: [Điền MSSV thành viên nhóm]

Nội dung:
1. Giới thiệu về LSTM và ưu điểm so với RNN đơn giản.
2. Chuẩn bị dữ liệu tên (tương tự HW6).
3. Xây dựng mô hình LSTM đơn giản từ đầu.
4. Huấn luyện mô hình LSTM.
5. Dự đoán ký tự và sinh tên bằng LSTM.

In [1]:
# -*- coding: utf-8 -*-
import numpy as np

## 1. Giới thiệu về LSTM

**Long Short-Term Memory (LSTM)** là một kiến trúc mạng nơ-ron hồi quy (RNN) đặc biệt, được thiết kế để giải quyết các vấn đề của RNN truyền thống, đặc biệt là vấn đề **vanishing gradient** (gradient biến mất) và khả năng học các **phụ thuộc xa (long-term dependencies)** trong dữ liệu chuỗi.

### Cấu trúc của một ô LSTM (LSTM Cell)

Điểm cốt lõi của LSTM là **ô nhớ (cell)**, có khả năng duy trì thông tin qua nhiều bước thời gian. Trạng thái của ô nhớ, gọi là **cell state ($C_t$)**, được điều khiển bởi ba loại cổng (gates):

1.  **Cổng Quên (Forget Gate - $f_t$):** Quyết định thông tin nào từ cell state trước đó ($C_{t-1}$) sẽ bị loại bỏ. Nó xem xét $h_{t-1}$ (output của bước trước) và $x_t$ (input hiện tại), và đưa ra một số từ 0 đến 1 cho mỗi số trong cell state $C_{t-1}$. 1 nghĩa là "giữ hoàn toàn", 0 nghĩa là "quên hoàn toàn".
    $f_t = \sigma(W_f \cdot [h_{t-1}, x_t] + b_f)$

2.  **Cổng Vào (Input Gate - $i_t$):** Quyết định thông tin mới nào sẽ được lưu trữ trong cell state. Bao gồm hai phần:
    * Một lớp sigmoid ($i_t$) quyết định giá trị nào sẽ được cập nhật.
        $i_t = \sigma(W_i \cdot [h_{t-1}, x_t] + b_i)$
    * Một lớp tanh ($\tilde{C}_t$) tạo ra một vector các giá trị ứng viên mới có thể được thêm vào cell state.
        $\tilde{C}_t = \tanh(W_C \cdot [h_{t-1}, x_t] + b_C)$

3.  **Cập nhật Cell State:** Cell state cũ $C_{t-1}$ được cập nhật thành $C_t$:
    $C_t = f_t * C_{t-1} + i_t * \tilde{C}_t$
    (Phần cũ được nhân với $f_t$, phần ứng viên mới được nhân với $i_t$).

4.  **Cổng Đầu Ra (Output Gate - $o_t$):** Quyết định phần nào của cell state sẽ được đưa ra làm output (hidden state $h_t$).
    * Một lớp sigmoid ($o_t$) quyết định phần nào của cell state sẽ được output.
        $o_t = \sigma(W_o \cdot [h_{t-1}, x_t] + b_o)$
    * Cell state được đưa qua hàm $\tanh$ (để giá trị nằm trong khoảng [-1, 1]) và sau đó nhân với output của cổng sigmoid.
        $h_t = o_t * \tanh(C_t)$

Trong đó, $\sigma$ là hàm sigmoid, và $[h_{t-1}, x_t]$ là phép nối (concatenation) của hidden state trước đó và input hiện tại. $W_f, W_i, W_C, W_o$ và $b_f, b_i, b_C, b_o$ là các ma trận trọng số và vector bias tương ứng được học trong quá trình huấn luyện.

### Ưu điểm của LSTM
-   Khả năng ghi nhớ thông tin trong thời gian dài tốt hơn RNN đơn giản.
-   Giảm thiểu vấn đề vanishing/exploding gradient.

(Tham khảo slide: "LSTM.pdf")

## 2. Chuẩn bị Dữ liệu (Tương tự HW6)

In [2]:
data_lstm = ["Bình", "Long", "Dũng"]

chars_lstm = set()
for name in data_lstm:
    for char in name:
        chars_lstm.add(char)

sorted_chars_lstm = sorted(list(chars_lstm))
char_to_int_lstm = {ch: i for i, ch in enumerate(sorted_chars_lstm)}
int_to_char_lstm = {i: ch for i, ch in enumerate(sorted_chars_lstm)}
vocab_size_lstm = len(sorted_chars_lstm)

print(f"Bộ từ vựng LSTM ({vocab_size_lstm} ký tự): {sorted_chars_lstm}")
print(f"Ánh xạ ký tự sang số (LSTM): {char_to_int_lstm}")

Bộ từ vựng LSTM (9 ký tự): ['B', 'D', 'L', 'g', 'h', 'n', 'o', 'ì', 'ũ']
Ánh xạ ký tự sang số (LSTM): {'B': 0, 'D': 1, 'L': 2, 'g': 3, 'h': 4, 'n': 5, 'o': 6, 'ì': 7, 'ũ': 8}


## 3. Xây dựng Mô hình LSTM

In [3]:
# Hàm kích hoạt
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh_activation(x): # Đổi tên để tránh trùng với np.tanh nếu dùng
    return np.tanh(x)

def softmax_activation(x): # Đổi tên
    e_x = np.exp(x - np.max(x, axis=0, keepdims=True))
    return e_x / np.sum(e_x, axis=0, keepdims=True)

class SimpleLSTM:
    def __init__(self, vocab_size, hidden_size, learning_rate=0.01):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.lr = learning_rate
        
        # Kích thước của vector input sau khi nối [h_prev, x_t]
        concat_size = hidden_size + vocab_size

        # Trọng số và bias cho các cổng và cell state
        # Cổng Quên (Forget Gate)
        self.W_f = np.random.randn(hidden_size, concat_size) * 0.01
        self.b_f = np.zeros((hidden_size, 1))
        # Cổng Vào (Input Gate)
        self.W_i = np.random.randn(hidden_size, concat_size) * 0.01
        self.b_i = np.zeros((hidden_size, 1))
        # Cell State Candidate
        self.W_c = np.random.randn(hidden_size, concat_size) * 0.01
        self.b_c = np.zeros((hidden_size, 1))
        # Cổng Đầu Ra (Output Gate)
        self.W_o = np.random.randn(hidden_size, concat_size) * 0.01
        self.b_o = np.zeros((hidden_size, 1))
        
        # Trọng số và bias cho lớp output (từ hidden state ra dự đoán ký tự)
        self.W_y = np.random.randn(vocab_size, hidden_size) * 0.01
        self.b_y = np.zeros((vocab_size, 1))

    def forward_step(self, x_one_hot, h_prev, C_prev):
        """
        Thực hiện một bước lan truyền xuôi qua ô LSTM.
        Args:
            x_one_hot (np.array): Vector one-hot của input hiện tại, shape (vocab_size, 1).
            h_prev (np.array): Hidden state từ bước trước, shape (hidden_size, 1).
            C_prev (np.array): Cell state từ bước trước, shape (hidden_size, 1).
        Returns:
            y_pred_proba, h_next, C_next, cache (các giá trị trung gian cho BPTT)
        """
        # Nối h_prev và x_one_hot
        concat_input = np.vstack((h_prev, x_one_hot)) # Shape (hidden_size + vocab_size, 1)
        
        # Cổng Quên
        f_t = sigmoid(np.dot(self.W_f, concat_input) + self.b_f)
        # Cổng Vào
        i_t = sigmoid(np.dot(self.W_i, concat_input) + self.b_i)
        # Cell State Candidate
        C_tilde_t = tanh_activation(np.dot(self.W_c, concat_input) + self.b_c)
        
        # Cell State mới
        C_next = f_t * C_prev + i_t * C_tilde_t
        
        # Cổng Đầu Ra
        o_t = sigmoid(np.dot(self.W_o, concat_input) + self.b_o)
        # Hidden State mới
        h_next = o_t * tanh_activation(C_next)
        
        # Lớp Output
        o_logits = np.dot(self.W_y, h_next) + self.b_y
        y_pred_proba = softmax_activation(o_logits)
        
        cache = {
            'concat_input': concat_input, 'f_t': f_t, 'i_t': i_t, 'C_tilde_t': C_tilde_t,
            'C_prev': C_prev, 'C_next': C_next, 'o_t': o_t, 'h_prev': h_prev, 'h_next': h_next,
            'x_one_hot': x_one_hot, 'o_logits': o_logits, 'y_pred_proba': y_pred_proba
        }
        return y_pred_proba, h_next, C_next, cache

    def train_sequence(self, input_char_indices, target_char_indices):
        loss = 0
        h_prev = np.zeros((self.hidden_size, 1))
        C_prev = np.zeros((self.hidden_size, 1))
        
        caches = [] # Lưu cache của từng bước thời gian

        # --- Lan truyền xuôi ---
        for t in range(len(input_char_indices)):
            x_idx = input_char_indices[t]
            y_target_idx = target_char_indices[t]
            
            x_one_hot = np.zeros((self.vocab_size, 1))
            x_one_hot[x_idx] = 1
            
            y_pred_proba_t, h_next_t, C_next_t, cache_t = self.forward_step(x_one_hot, h_prev, C_prev)
            caches.append(cache_t)
            
            loss_t = -np.log(y_pred_proba_t[y_target_idx, 0] + 1e-9)
            loss += loss_t
            
            h_prev, C_prev = h_next_t, C_next_t
            
        avg_loss = loss / len(input_char_indices)

        # --- Lan truyền ngược (BPTT cho LSTM) ---
        # Khởi tạo gradients
        dW_f, dW_i, dW_c, dW_o, dW_y = (np.zeros_like(p) for p in [self.W_f, self.W_i, self.W_c, self.W_o, self.W_y])
        db_f, db_i, db_c, db_o, db_y = (np.zeros_like(p) for p in [self.b_f, self.b_i, self.b_c, self.b_o, self.b_y])
        
        dh_next_bp = np.zeros_like(h_prev) # Gradient của loss theo h_t từ bước t+1
        dC_next_bp = np.zeros_like(C_prev) # Gradient của loss theo C_t từ bước t+1

        for t in reversed(range(len(input_char_indices))):
            cache_t = caches[t]
            y_target_idx = target_char_indices[t]
            
            # Gradient của loss theo output logits (dL/do_logits)
            dy_logits = np.copy(cache_t['y_pred_proba'])
            dy_logits[y_target_idx] -= 1
            
            # Gradients cho W_y, b_y
            dW_y += np.dot(dy_logits, cache_t['h_next'].T)
            db_y += dy_logits
            
            # Gradient của loss theo h_next[t] (lan truyền từ output layer và từ h_next_bp của bước t+1)
            dh_t = np.dot(self.W_y.T, dy_logits) + dh_next_bp
            
            # --- Backprop qua Output Gate và tanh(C_next) ---
            # dL/do_t = dL/dh_t * dh_t/do_t = dh_t * tanh(C_next[t])
            do_t = dh_t * tanh_activation(cache_t['C_next'])
            # d(sigmoid(z))/dz = sigmoid(z) * (1 - sigmoid(z))
            do_t_raw = do_t * cache_t['o_t'] * (1 - cache_t['o_t']) # dL/d(raw_input_to_o_gate)
            dW_o += np.dot(do_t_raw, cache_t['concat_input'].T)
            db_o += do_t_raw
            
            # Gradient của loss theo C_next[t] (lan truyền từ h_next[t] và từ C_next_bp của bước t+1)
            # dL/dC_next[t] = dL/dh_t * dh_t/dC_next[t] + dL/dC_next_bp[t+1]
            # dh_t/dC_next[t] = o_t[t] * (1 - tanh(C_next[t])^2)
            dC_t = dh_t * cache_t['o_t'] * (1 - tanh_activation(cache_t['C_next'])**2) + dC_next_bp
            
            # --- Backprop qua Cell State Update (C_t = f_t * C_{t-1} + i_t * C_tilde_t) ---
            # dL/dC_prev[t-1] = dL/dC_t * dC_t/dC_prev[t-1] = dC_t * f_t[t]
            dC_prev_t_minus_1 = dC_t * cache_t['f_t'] # Sẽ là dC_next_bp cho bước t-1

            # dL/df_t = dL/dC_t * dC_t/df_t = dC_t * C_prev[t-1] (C_prev của bước hiện tại, tức là C_next của bước t-1)
            df_t = dC_t * cache_t['C_prev']
            df_t_raw = df_t * cache_t['f_t'] * (1 - cache_t['f_t'])
            dW_f += np.dot(df_t_raw, cache_t['concat_input'].T)
            db_f += df_t_raw
            
            # dL/di_t = dL/dC_t * dC_t/di_t = dC_t * C_tilde_t[t]
            di_t = dC_t * cache_t['C_tilde_t']
            di_t_raw = di_t * cache_t['i_t'] * (1 - cache_t['i_t'])
            dW_i += np.dot(di_t_raw, cache_t['concat_input'].T)
            db_i += di_t_raw
            
            # dL/dC_tilde_t = dL/dC_t * dC_t/dC_tilde_t = dC_t * i_t[t]
            dC_tilde_t = dC_t * cache_t['i_t']
            # d(tanh(z))/dz = 1 - tanh(z)^2
            dC_tilde_t_raw = dC_tilde_t * (1 - cache_t['C_tilde_t']**2)
            dW_c += np.dot(dC_tilde_t_raw, cache_t['concat_input'].T)
            db_c += dC_tilde_t_raw
            
            # Gradient lan truyền về concat_input
            d_concat_input = (np.dot(self.W_f.T, df_t_raw) +
                              np.dot(self.W_i.T, di_t_raw) +
                              np.dot(self.W_c.T, dC_tilde_t_raw) +
                              np.dot(self.W_o.T, do_t_raw))
            
            # Cập nhật dh_next_bp và dC_next_bp cho bước lặp ngược tiếp theo (t-1)
            dh_next_bp = d_concat_input[:self.hidden_size, :] # Phần gradient của h_prev[t-1]
            dC_next_bp = dC_prev_t_minus_1 # Gradient của C_prev[t-1]

        # Gradient clipping
        gradients_list = [dW_f, dW_i, dW_c, dW_o, dW_y, db_f, db_i, db_c, db_o, db_y]
        for dparam in gradients_list:
            np.clip(dparam, -5, 5, out=dparam)
            
        gradients = {
            'dW_f': dW_f, 'dW_i': dW_i, 'dW_c': dW_c, 'dW_o': dW_o, 'dW_y': dW_y,
            'db_f': db_f, 'db_i': db_i, 'db_c': db_c, 'db_o': db_o, 'db_y': db_y
        }
        return avg_loss, gradients

    def update_parameters(self, gradients):
        self.W_f -= self.lr * gradients['dW_f']
        self.b_f -= self.lr * gradients['db_f']
        self.W_i -= self.lr * gradients['dW_i']
        self.b_i -= self.lr * gradients['db_i']
        self.W_c -= self.lr * gradients['dW_c']
        self.b_c -= self.lr * gradients['db_c']
        self.W_o -= self.lr * gradients['dW_o']
        self.b_o -= self.lr * gradients['db_o']
        self.W_y -= self.lr * gradients['dW_y']
        self.b_y -= self.lr * gradients['db_y']

## 4. Huấn luyện Mô hình LSTM

In [10]:
hidden_size_lstm = 70 # Có thể cần hidden_size lớn hơn RNN một chút
learning_rate_lstm = 0.01
n_epochs_lstm = 800 # LSTM có thể cần nhiều epochs hơn để học

lstm_model = SimpleLSTM(vocab_size_lstm, hidden_size_lstm, learning_rate_lstm)

print(f"\nBắt đầu huấn luyện LSTM với hidden_size={hidden_size_lstm}, lr={learning_rate_lstm}...")

for epoch in range(n_epochs_lstm):
    total_loss_epoch_lstm = 0
    for name_str in data_lstm:
        input_indices = [char_to_int_lstm[ch] for ch in name_str[:-1]]
        target_indices = [char_to_int_lstm[ch] for ch in name_str[1:]]
        
        if not input_indices: continue
            
        loss, grads = lstm_model.train_sequence(input_indices, target_indices)
        lstm_model.update_parameters(grads)
        total_loss_epoch_lstm += loss
        
    avg_loss_epoch_lstm = total_loss_epoch_lstm / len(data_lstm)
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{n_epochs_lstm}, Loss trung bình LSTM: {avg_loss_epoch_lstm:.4f}")

print("Hoàn tất huấn luyện LSTM.")


Bắt đầu huấn luyện LSTM với hidden_size=70, lr=0.01...
Epoch 100/800, Loss trung bình LSTM: 1.8359
Epoch 200/800, Loss trung bình LSTM: 1.7708
Epoch 300/800, Loss trung bình LSTM: 1.7391
Epoch 400/800, Loss trung bình LSTM: 1.7113
Epoch 500/800, Loss trung bình LSTM: 1.6714
Epoch 600/800, Loss trung bình LSTM: 1.6012
Epoch 700/800, Loss trung bình LSTM: 1.4904
Epoch 800/800, Loss trung bình LSTM: 1.3505
Hoàn tất huấn luyện LSTM.


## 5. Dự đoán Ký tự và Sinh Tên bằng LSTM

In [9]:
def complete_name_lstm(model, start_char_str, max_length=5):
    if start_char_str not in char_to_int_lstm:
        print(f"Ký tự bắt đầu '{start_char_str}' không có trong bộ từ vựng.")
        return start_char_str
        
    current_name = start_char_str
    h = np.zeros((model.hidden_size, 1)) # Hidden state ban đầu
    C = np.zeros((model.hidden_size, 1)) # Cell state ban đầu
    
    current_char_idx = char_to_int_lstm[start_char_str]

    for _ in range(max_length - len(start_char_str)):
        x_one_hot = np.zeros((model.vocab_size, 1))
        x_one_hot[current_char_idx] = 1
        
        y_pred_proba, h_next, C_next, _ = model.forward_step(x_one_hot, h, C)
        
        next_char_idx = np.argmax(y_pred_proba.flatten())
        next_char = int_to_char_lstm[next_char_idx]
        current_name += next_char
        
        current_char_idx = next_char_idx
        h, C = h_next, C_next
        
        if len(current_name) >= len(data_lstm[0]) + 2: break
            
    return current_name

print("\n--- Dự đoán và Hoàn thành Tên (LSTM) ---")
target_names_lstm = ["Bình", "Long", "Dũng"]
start_chars_lstm = ["B", "L", "D"]

for i, start_char in enumerate(start_chars_lstm):
    completed_lstm = complete_name_lstm(lstm_model, start_char, max_length=len(target_names_lstm[i]))
    print(f"Bắt đầu bằng '{start_char}', Hoàn thành (LSTM): '{completed_lstm}' (Mong muốn: '{target_names_lstm[i]}')")


--- Dự đoán và Hoàn thành Tên (LSTM) ---
Bắt đầu bằng 'B', Hoàn thành (LSTM): 'Bngn' (Mong muốn: 'Bình')
Bắt đầu bằng 'L', Hoàn thành (LSTM): 'Lngn' (Mong muốn: 'Long')
Bắt đầu bằng 'D', Hoàn thành (LSTM): 'Dngn' (Mong muốn: 'Dũng')
